# Practical Statistics for Data Scientists (Python)
# Chapter 1 — Exploratory Data Analysis

> **Author:** Satrio Arjuna Putra  
> **Reference:** *Practical Statistics for Data Scientists* — Bruce, Bruce & Gedeck (O'Reilly)

---

## 📋 Chapter Summary

Chapter 1 introduces **Exploratory Data Analysis (EDA)** — the process of summarising, visualising, and understanding a dataset *before* formal modelling. Key topics include:

| Topic | Key Concepts |
|-------|-------------|
| **Estimates of Location** | Mean, trimmed mean, weighted mean, median |
| **Estimates of Variability** | Std deviation, variance, IQR, MAD |
| **Exploring the Distribution** | Percentiles, boxplots, frequency tables, histograms, density plots |
| **Exploring Binary/Categorical Data** | Mode, bar charts, pie charts |
| **Exploring Relationships** | Correlation, scatter plots, heat maps, hexbin plots |

EDA helps detect outliers, spot patterns, test assumptions, and guide model selection.

---


## 🔧 Setup & Imports

We import the required libraries. Data is loaded directly from the official book repository via URL so no local files are needed.


In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
from scipy.stats import trim_mean
from statsmodels import robust
import wquantiles

import seaborn as sns
import matplotlib.pylab as plt

# Load datasets directly from the official book repository
DATA_BASE = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/"

STATE_CSV        = DATA_BASE + "state.csv"
AIRLINE_STATS    = DATA_BASE + "airline_stats.csv"
KC_TAX_CSV       = DATA_BASE + "kc_tax.csv.gz"
LC_LOANS_CSV     = DATA_BASE + "lc_loans.csv"
AIRPORT_DELAYS   = DATA_BASE + "dfw_airline.csv"
SP500_DATA       = DATA_BASE + "sp500_data.csv.gz"
SP500_SECTORS    = DATA_BASE + "sp500_sectors.csv"

print("Libraries loaded successfully!")


---

## 1. Estimates of Location

### 📚 Theory

**Location estimates** summarise where the data is *centred*.

| Measure | Formula | Notes |
|---------|---------|-------|
| **Mean** | $\bar{x} = \frac{1}{n}\sum x_i$ | Sensitive to outliers |
| **Trimmed Mean** | Remove top/bottom p%, then mean | More robust |
| **Weighted Mean** | $\bar{x}_w = \frac{\sum w_i x_i}{\sum w_i}$ | Accounts for importance |
| **Median** | Middle value when sorted | Robust to outliers |

> **Key Insight:** Use the *median* or *trimmed mean* when data has outliers. Use the *weighted mean* when some observations are more important than others.


In [ ]:
# Load state population data
state = pd.read_csv(STATE_CSV)
print("Shape:", state.shape)
print(state.head(8))


In [ ]:
# Mean of Population
print("Mean Population:", state['Population'].mean())

# Trimmed mean (remove bottom/top 10%)
print("Trimmed Mean Population:", trim_mean(state['Population'], 0.1))

# Median
print("Median Population:", state['Population'].median())


In [ ]:
# Murder rate statistics
print("Mean Murder Rate:", state['Murder.Rate'].mean())
print("Weighted Mean Murder Rate (by Population):", np.average(state['Murder.Rate'], weights=state['Population']))
print("Weighted Median Murder Rate:", wquantiles.median(state['Murder.Rate'], weights=state['Population']))


---

## 2. Estimates of Variability

### 📚 Theory

**Variability** (or **spread**) measures how dispersed the data is.

| Measure | Description |
|---------|------------|
| **Variance / Std Dev** | Average squared deviation from the mean |
| **Mean Absolute Deviation** | Average absolute deviation from the mean |
| **Median Absolute Deviation (MAD)** | Robust alternative to std dev |
| **IQR** | Q3 - Q1; spread of the middle 50% |
| **Range** | Max - Min; sensitive to outliers |

> **Key Insight:** Standard deviation is the most common but is sensitive to outliers. IQR and MAD are more robust alternatives.


In [ ]:
# Standard deviation
print("Std Dev Population:", state['Population'].std())

# IQR
q75, q25 = state['Population'].quantile(0.75), state['Population'].quantile(0.25)
print("IQR:", q75 - q25)

# Median Absolute Deviation
print("MAD:", robust.scale.mad(state['Population']))


---

## 3. Exploring the Data Distribution

### 📚 Theory

Beyond location and spread, we study the **shape** of the distribution:
- **Percentiles / Quantiles:** Value below which a given % of data falls
- **Boxplots:** Visualise Q1, median, Q3, and outliers
- **Frequency Table / Histogram:** Count observations in bins
- **Density Plot (KDE):** Smooth estimate of the probability density

> **Key Insight:** Histograms and density plots reveal skewness, modes, and gaps that summary statistics hide.


In [ ]:
# Percentiles of Murder Rate
percentages = [0.05, 0.25, 0.5, 0.75, 0.95]
print(state['Murder.Rate'].quantile(percentages))


In [ ]:
# Boxplot of Population
fig, ax = plt.subplots(figsize=(4, 6))
ax = (state['Population'] / 1_000_000).plot.box(ax=ax)
ax.set_ylabel('Population (millions)')
ax.set_title('Boxplot: US State Populations')
plt.tight_layout()
plt.show()


In [ ]:
# Histogram of Population
ax = (state['Population'] / 1_000_000).plot.hist(figsize=(6, 4), bins=10)
ax.set_xlabel('Population (millions)')
ax.set_title('Histogram: US State Populations')
plt.tight_layout()
plt.show()


In [ ]:
# Density plot (KDE) — Murder Rate
state['Murder.Rate'].plot.density(figsize=(6, 4))
plt.xlabel('Murder Rate per 100,000')
plt.title('Density Plot: Murder Rates')
plt.tight_layout()
plt.show()


---

## 4. Exploring Binary and Categorical Data

### 📚 Theory

For **categorical variables**, we summarise using:
- **Mode:** Most frequent category
- **Bar chart:** Frequency of each category
- **Pie chart:** Proportion of each category

> **Key Insight:** Bar charts are generally preferred over pie charts because humans are better at judging length than angle.


In [ ]:
# Airport delays — categorical example
dfw = pd.read_csv(AIRPORT_DELAYS)
print(dfw.head())
print("\nColumn sums:")
print(100 * dfw / dfw.values.sum())


In [ ]:
# Bar chart of airport delays
dfw = pd.read_csv(AIRPORT_DELAYS)
dfw.transpose().plot.bar(figsize=(8, 4), legend=False)
plt.ylabel('Count of Delays')
plt.title('DFW Airport Delay Causes')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


---

## 5. Exploring Relationships Between Variables

### 📚 Theory

To explore **relationships between two variables**:
- **Scatter plot:** Raw x–y relationship
- **Hexbin / Contour:** For dense datasets (avoids overplotting)
- **Correlation matrix / Heat map:** Pairwise linear relationships (-1 to +1)

**Correlation coefficient (Pearson):**
$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{(n-1)s_x s_y}$$

> **Key Insight:** Correlation measures *linear* association. A correlation of 0 does NOT mean no relationship — it means no *linear* relationship.


In [ ]:
# S&P 500 data — correlation between stocks
sp500_px  = pd.read_csv(SP500_DATA, index_col=0)
sp500_sym = pd.read_csv(SP500_SECTORS)
sp500_sym = sp500_sym[sp500_sym['sector'] == 'Technology']
tech_stocks = sp500_px[sp500_sym['symbol']]

# Pairwise correlation
corr_matrix = tech_stocks.corr()
print("Correlation matrix (top 5x5):")
print(corr_matrix.iloc[:5, :5].round(2))


In [ ]:
# Hexbin plot: two variables (avoids overplotting)
kc_tax = pd.read_csv(KC_TAX_CSV)
kc_tax_filtered = kc_tax.loc[
    (kc_tax.TaxAssessedValue < 750000) &
    (kc_tax.SqFtTotLiving > 100) &
    (kc_tax.SqFtTotLiving < 3500), :]

ax = kc_tax_filtered.plot.hexbin(
    x='SqFtTotLiving', y='TaxAssessedValue',
    gridsize=30, figsize=(8, 5), colormap='Blues')
ax.set_xlabel('Living Area (sq ft)')
ax.set_ylabel('Tax Assessed Value ($)')
ax.set_title('KC House Data: Living Area vs Tax Value')
plt.tight_layout()
plt.show()


In [ ]:
# Violin + boxplots: categorical vs numeric
lc_loans = pd.read_csv(LC_LOANS_CSV)
ax = lc_loans.boxplot(by='grade', column='loan_amnt', figsize=(10, 4))
ax.set_xlabel('Loan Grade')
ax.set_ylabel('Loan Amount ($)')
ax.set_title('Loan Amount by Grade')
plt.suptitle('')
plt.tight_layout()
plt.show()


---

## ✅ Chapter 1 Key Takeaways

1. **Mean vs Median:** Use median (or trimmed mean) when data has outliers; mean when data is roughly symmetric.
2. **Std Dev vs IQR:** IQR and MAD are robust measures of spread; std dev is sensitive to outliers.
3. **Visualisation matters:** Summary statistics alone can miss skewness, multi-modality, and outliers.
4. **Correlations ≠ Causation:** Correlation measures linear association only; always visualise before interpreting.
5. **EDA is iterative:** You generate questions, explore, refine, and repeat.

---
*Reference: Chapter 1, Practical Statistics for Data Scientists (O'Reilly)*
